# TPC-H Benchmark Queries

**Dataset:** `samples.tpch (all tables)`

**Difficulty:** Hard

**Topics:** classic TPC-H patterns, self-joins, exists/not-exists patterns, complex aggregations

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

orders   = spark.table("samples.tpch.orders")
lineitem = spark.table("samples.tpch.lineitem")
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")
region   = spark.table("samples.tpch.region")
part     = spark.table("samples.tpch.part")
supplier = spark.table("samples.tpch.supplier")
partsupp = spark.table("samples.tpch.partsupp")

## Problem 1

**Parts with Highest Total Available Value.**

Join `partsupp` with `part` (on `ps_partkey = p_partkey`). Compute `total_available_value = sum(ps_availqty * ps_supplycost)` per part, along with the number of distinct suppliers for that part. Return the top 20 parts by total available value descending.

Business context: Inventory valuation at the part level helps the warehouse finance team identify which parts represent the largest capital tie-up — informing decisions about order frequency and safety-stock levels.

**Expected output columns:** `p_partkey`, `p_name`, `p_brand`, `total_available_value`, `supplier_count`

In [ ]:
# Problem 1 — write your solution here
# Assign your result to: result_1
#   joined = partsupp.join(part, partsupp.ps_partkey == part.p_partkey)
#   result_1 = joined.groupBy('p_partkey', 'p_name', 'p_brand').agg(
#       F.sum(F.col('ps_availqty') * F.col('ps_supplycost')).alias('total_available_value'),
#       F.countDistinct('ps_suppkey').alias('supplier_count')
#   ).orderBy(F.desc('total_available_value')).limit(20)

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None — did you assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
for col in ['p_partkey', 'p_name', 'p_brand', 'total_available_value', 'supplier_count']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt == 20, f"Expected exactly 20 rows (top 20 parts), got {cnt}"
assert result_1.agg(F.min('total_available_value')).collect()[0][0] > 0, "total_available_value must be positive"
assert result_1.agg(F.min('supplier_count')).collect()[0][0] > 0, "supplier_count must be positive"
values = [r['total_available_value'] for r in result_1.orderBy('total_available_value', ascending=False).collect()]
assert values == sorted(values, reverse=True), "Results should be sorted by total_available_value descending"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

**Supplier Market Share per Nation.**

Join `supplier` → `partsupp` → `nation`. Compute each supplier's revenue contribution as `ps_supplycost * ps_availqty`, then express it as a percentage of their nation's total. Return the top 5 suppliers per nation by market share.

Business context: Nation-level supplier concentration analysis helps procurement teams identify single-source dependency risk — if one supplier controls >30% of a nation's supply, that creates significant supply-chain vulnerability.


**Expected output columns:** `n_name`, `s_name`, `supplier_revenue`, `nation_total`, `market_share_pct`

In [ ]:
# Problem 2 — write your solution here
# Assign your result to: result_2
#   joined = supplier.join(nation, supplier.s_nationkey == nation.n_nationkey)\
#                    .join(partsupp, supplier.s_suppkey == partsupp.ps_suppkey)
#   agg = joined.groupBy('s_suppkey', 's_name', 'n_name').agg(
#       F.sum(F.col('ps_supplycost') * F.col('ps_availqty')).alias('supplier_revenue')
#   )
#   w = Window.partitionBy('n_name')
#   agg = agg.withColumn('nation_total', F.sum('supplier_revenue').over(w))\
#            .withColumn('market_share_pct', F.col('supplier_revenue') / F.col('nation_total') * 100)
#   rank_w = Window.partitionBy('n_name').orderBy(F.desc('supplier_revenue'))
#   result_2 = agg.withColumn('rn', F.rank().over(rank_w)).filter(F.col('rn') <= 5)\
#              .select('n_name', 's_name', 'supplier_revenue', 'nation_total', 'market_share_pct')

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None — did you assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
for col in ['n_name', 's_name', 'supplier_revenue', 'nation_total', 'market_share_pct']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_2.agg(F.min('supplier_revenue')).collect()[0][0] > 0, "supplier_revenue must be positive"
assert result_2.agg(F.min('market_share_pct')).collect()[0][0] > 0, "market_share_pct must be positive"
assert result_2.agg(F.max('market_share_pct')).collect()[0][0] <= 100, "market_share_pct cannot exceed 100"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

**Late Delivery Analysis.**

In `lineitem`, find line items where `l_receiptdate > l_commitdate` (late delivery). Compute per `l_shipmode`: total items shipped, number of late items, number of on-time items, and `late_rate_pct = late_items / total_items * 100`.

Business context: Late delivery analysis by shipping mode helps the logistics team identify which carriers or transport methods are underperforming against SLA, feeding into annual carrier renegotiation reviews.

**Expected output columns:** `l_shipmode`, `total_items`, `late_items`, `on_time_items`, `late_rate_pct`

In [ ]:
# Problem 3 — write your solution here
# Assign your result to: result_3
#   result_3 = lineitem.groupBy('l_shipmode').agg(
#       F.count('*').alias('total_items'),
#       F.sum(F.when(F.col('l_receiptdate') > F.col('l_commitdate'), 1).otherwise(0)).alias('late_items'),
#       F.sum(F.when(F.col('l_receiptdate') <= F.col('l_commitdate'), 1).otherwise(0)).alias('on_time_items')
#   ).withColumn('late_rate_pct', F.col('late_items') / F.col('total_items') * 100)

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None — did you assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
for col in ['l_shipmode', 'total_items', 'late_items', 'on_time_items', 'late_rate_pct']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
# late + on_time should equal total
check = result_3.withColumn('sum_check', F.col('late_items') + F.col('on_time_items'))\
                .filter(F.col('sum_check') != F.col('total_items')).count()
assert check == 0, f"Found {check} rows where late_items + on_time_items != total_items"
assert result_3.agg(F.min('late_rate_pct')).collect()[0][0] >= 0, "late_rate_pct must be non-negative"
assert result_3.agg(F.max('late_rate_pct')).collect()[0][0] <= 100, "late_rate_pct cannot exceed 100"
print(f"Problem 3 passed ✓  ({cnt} ship modes)")

## Problem 4

**Customer Order Gap Analysis.**

For each customer, compute the average number of days between consecutive orders using a lag window function. Join with the `customer` table to include name and market segment. Sort by average days between orders descending to find customers with the longest gaps.

Business context: Customers with large order gaps who previously ordered frequently may be churning. The retention team uses this analysis to trigger proactive outreach campaigns before they defect to a competitor.


**Expected output columns:** `c_custkey`, `c_name`, `c_mktsegment`, `order_count`, `avg_days_between_orders`

In [ ]:
# Problem 4 — write your solution here
# Assign your result to: result_4
#   w = Window.partitionBy('o_custkey').orderBy('o_orderdate')
#   with_lag = orders.withColumn('prev_order_date', F.lag('o_orderdate', 1).over(w))
#   with_gap = with_lag.withColumn('days_gap',
#       F.datediff(F.col('o_orderdate'), F.col('prev_order_date')))
#   agg = with_gap.groupBy('o_custkey').agg(
#       F.count('o_orderkey').alias('order_count'),
#       F.avg('days_gap').alias('avg_days_between_orders')
#   ).filter(F.col('avg_days_between_orders').isNotNull())
#   result_4 = agg.join(customer, agg.o_custkey == customer.c_custkey)\
#              .select('c_custkey', 'c_name', 'c_mktsegment', 'order_count', 'avg_days_between_orders')\
#              .orderBy(F.desc('avg_days_between_orders'))

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None — did you assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
for col in ['c_custkey', 'c_name', 'c_mktsegment', 'order_count', 'avg_days_between_orders']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_4.agg(F.min('order_count')).collect()[0][0] > 0, "order_count must be positive"
assert result_4.agg(F.min('avg_days_between_orders')).collect()[0][0] >= 0, "avg_days_between_orders must be non-negative"
null_gaps = result_4.filter(F.col('avg_days_between_orders').isNull()).count()
assert null_gaps == 0, f"Found {null_gaps} rows with null avg_days_between_orders"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

**Part Substitution Analysis.**

Find `part` records of the same `p_brand` and `p_type` that are supplied by multiple suppliers (via `partsupp`). For these parts, show the range of supply costs: min, max, and `cost_range = max - min`. Include the count of distinct parts and distinct suppliers.

Business context: Parts with the same brand and type but wide cost ranges indicate supplier price disparity — a signal that procurement should consolidate to the lowest-cost supplier or renegotiate with higher-cost ones.

**Expected output columns:** `p_brand`, `p_type`, `part_count`, `supplier_count`, `min_supply_cost`, `max_supply_cost`, `cost_range`

In [ ]:
# Problem 5 — write your solution here
# Assign your result to: result_5
#   joined = part.join(partsupp, part.p_partkey == partsupp.ps_partkey)
#   result_5 = joined.groupBy('p_brand', 'p_type').agg(
#       F.countDistinct('p_partkey').alias('part_count'),
#       F.countDistinct('ps_suppkey').alias('supplier_count'),
#       F.min('ps_supplycost').alias('min_supply_cost'),
#       F.max('ps_supplycost').alias('max_supply_cost')
#   ).filter(F.col('supplier_count') > 1)\
#    .withColumn('cost_range', F.col('max_supply_cost') - F.col('min_supply_cost'))\
#    .orderBy(F.desc('cost_range'))

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None — did you assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
for col in ['p_brand', 'p_type', 'part_count', 'supplier_count',
            'min_supply_cost', 'max_supply_cost', 'cost_range']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 7, f"Expected exactly 7 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_5.agg(F.min('supplier_count')).collect()[0][0] > 1, "All groups should have > 1 supplier"
assert result_5.agg(F.min('cost_range')).collect()[0][0] >= 0, "cost_range must be non-negative"
# max >= min
invalid = result_5.filter(F.col('max_supply_cost') < F.col('min_supply_cost')).count()
assert invalid == 0, f"Found {invalid} rows where max_supply_cost < min_supply_cost"
print(f"Problem 5 passed ✓  ({cnt} brand-type groups)")

## Problem 6

**High-Value Customer Segments.**

Compute each customer's total spend from the `orders` table (`sum(o_totalprice)`). Use `ntile(4)` window function to classify customers into spend quartiles (1=lowest, 4=highest). Show the distribution of customer count and total spend per quartile per `c_mktsegment`.

Business context: Quartile segmentation helps the sales team size their accounts — quartile 4 customers in each segment receive dedicated account management, while quartile 1 customers are handled through self-serve channels.


**Expected output columns:** `c_mktsegment`, `spend_quartile`, `customer_count`, `total_spend`, `avg_spend`

In [ ]:
# Problem 6 — write your solution here
# Assign your result to: result_6
#   cust_spend = orders.groupBy('o_custkey').agg(F.sum('o_totalprice').alias('total_spend'))
#   w = Window.orderBy('total_spend')
#   with_quartile = cust_spend.withColumn('spend_quartile', F.ntile(4).over(w))
#   with_segment = with_quartile.join(customer, with_quartile.o_custkey == customer.c_custkey)
#   result_6 = with_segment.groupBy('c_mktsegment', 'spend_quartile').agg(
#       F.count('c_custkey').alias('customer_count'),
#       F.sum('total_spend').alias('total_spend'),
#       F.avg('total_spend').alias('avg_spend')
#   ).orderBy('c_mktsegment', 'spend_quartile')

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None — did you assign your DataFrame?"
assert hasattr(result_6, 'columns'), "result_6 must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
for col in ['c_mktsegment', 'spend_quartile', 'customer_count', 'total_spend', 'avg_spend']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 5, f"Expected exactly 5 columns, got {len(cols)}: {cols}"
cnt = result_6.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
quartiles = result_6.select('spend_quartile').distinct().collect()
quartile_vals = {r['spend_quartile'] for r in quartiles}
assert quartile_vals == {1, 2, 3, 4}, f"Expected quartiles 1-4, got {quartile_vals}"
assert result_6.agg(F.min('customer_count')).collect()[0][0] > 0, "customer_count must be positive"
print(f"Problem 6 passed ✓  ({cnt} rows)")

## Problem 7

**Order Fulfilment Time.**

Compute fulfilment time per order as `min(l_receiptdate) - o_orderdate` (in days). Join with `customer` to get market segment. Compute average fulfilment days per `c_mktsegment` and `o_orderpriority`, along with the order count.

Business context: SLA compliance analysis by market segment and priority tier reveals whether the fulfilment team is honouring priority commitments — high-priority orders should have materially shorter fulfilment times than low-priority ones.

**Expected output columns:** `c_mktsegment`, `o_orderpriority`, `avg_fulfilment_days`, `order_count`

In [ ]:
# Problem 7 — write your solution here
# Assign your result to: result_7
#   receipt_by_order = lineitem.groupBy('l_orderkey').agg(
#       F.min('l_receiptdate').alias('min_receipt_date'))
#   with_fulfil = orders.join(receipt_by_order, orders.o_orderkey == receipt_by_order.l_orderkey)\
#                       .withColumn('fulfilment_days',
#                           F.datediff(F.col('min_receipt_date'), F.col('o_orderdate')))
#   with_cust = with_fulfil.join(customer, with_fulfil.o_custkey == customer.c_custkey)
#   result_7 = with_cust.groupBy('c_mktsegment', 'o_orderpriority').agg(
#       F.avg('fulfilment_days').alias('avg_fulfilment_days'),
#       F.count('o_orderkey').alias('order_count')
#   ).orderBy('c_mktsegment', 'o_orderpriority')

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None — did you assign your DataFrame?"
assert hasattr(result_7, 'columns'), "result_7 must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
for col in ['c_mktsegment', 'o_orderpriority', 'avg_fulfilment_days', 'order_count']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_7.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_7.agg(F.min('avg_fulfilment_days')).collect()[0][0] > 0, "avg_fulfilment_days must be positive"
assert result_7.agg(F.min('order_count')).collect()[0][0] > 0, "order_count must be positive"
print(f"Problem 7 passed ✓  ({cnt} rows)")

## Problem 8

**Revenue at Risk from Returns.**

Identify returned line items (`l_returnflag = 'R'`). Compute revenue at risk as `sum(l_extendedprice)` for returned items per nation. Join `lineitem` → `orders` → `customer` → `nation`. Also compute total revenue (all items) per nation, and `return_rate_pct = returned_revenue / total_revenue * 100`. Sort descending by returned revenue.

Business context: High return rates in specific nations may indicate product-quality issues, mis-selling, or fulfilment problems. The returns management team uses this to prioritise root-cause investigations by market.

**Expected output columns:** `n_name`, `returned_revenue`, `total_revenue`, `return_rate_pct`

In [ ]:
# Problem 8 — write your solution here
# Assign your result to: result_8
#   joined = lineitem.join(orders, lineitem.l_orderkey == orders.o_orderkey)\
#                    .join(customer, orders.o_custkey == customer.c_custkey)\
#                    .join(nation, customer.c_nationkey == nation.n_nationkey)
#   result_8 = joined.groupBy('n_name').agg(
#       F.sum(F.when(F.col('l_returnflag') == 'R', F.col('l_extendedprice')).otherwise(0)).alias('returned_revenue'),
#       F.sum('l_extendedprice').alias('total_revenue')
#   ).withColumn('return_rate_pct', F.col('returned_revenue') / F.col('total_revenue') * 100)\
#    .orderBy(F.desc('returned_revenue'))

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None — did you assign your DataFrame?"
assert hasattr(result_8, 'columns'), "result_8 must be a Spark DataFrame"
cols = [c.lower() for c in result_8.columns]
for col in ['n_name', 'returned_revenue', 'total_revenue', 'return_rate_pct']:
    assert col in cols, f"Missing column: {col}"
assert len(cols) == 4, f"Expected exactly 4 columns, got {len(cols)}: {cols}"
cnt = result_8.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
assert result_8.agg(F.min('total_revenue')).collect()[0][0] > 0, "total_revenue must be positive"
assert result_8.agg(F.min('returned_revenue')).collect()[0][0] >= 0, "returned_revenue must be non-negative"
assert result_8.agg(F.max('return_rate_pct')).collect()[0][0] <= 100, "return_rate_pct cannot exceed 100"
# returned_revenue <= total_revenue
invalid = result_8.filter(F.col('returned_revenue') > F.col('total_revenue')).count()
assert invalid == 0, f"Found {invalid} rows where returned_revenue > total_revenue"
print(f"Problem 8 passed ✓  ({cnt} nations)")